#### **Import Libraries & Configure Environment**

In [1]:
import pandas as pd
import numpy as np

# Display settings for better readability
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

print("Libraries imported successfully!")

Libraries imported successfully!


#### **Load Raw Datasets & Define Country Cohort**

In [2]:
# Define the target 10-country
TARGET_COUNTRIES = [
    "Ethiopia", "Kenya", "Tanzania", "Uganda", "Rwanda", 
    "Ghana", "Nigeria", "South Africa", "Egypt", "Morocco"
]

print(f"Target Country Cohort ({len(TARGET_COUNTRIES)} countries):")
print(TARGET_COUNTRIES)

# 1. Load Ember Yearly Data
ember_raw = pd.read_csv("../data/raw/ember_yearly_electricity.csv")
print("\nRaw Ember Shape:", ember_raw.shape)

# 2. Inspect the first 5 lines of the World Bank file to check for metadata headers
print("\n--- Inspecting World Bank File Header Lines ---")
with open("../data/raw/world_bank_development_indicators.csv", "r", encoding="utf-8") as f:
    for i in range(5):
        print(f"Line {i+1}: {f.readline().strip()}")

# 3. Load World Bank data
try:
    wb_raw = pd.read_csv("../data/raw/world_bank_development_indicators.csv", skiprows=4)
    print("\nRaw World Bank Shape (with skiprows=4):", wb_raw.shape)
except Exception as e:
    print("\nTrying default load due to error:", e)
    wb_raw = pd.read_csv("../data/raw/world_bank_development_indicators.csv", on_bad_lines='skip')
    print("Raw World Bank Shape (fallback):", wb_raw.shape)

Target Country Cohort (10 countries):
['Ethiopia', 'Kenya', 'Tanzania', 'Uganda', 'Rwanda', 'Ghana', 'Nigeria', 'South Africa', 'Egypt', 'Morocco']

Raw Ember Shape: (104203, 24)

--- Inspecting World Bank File Header Lines ---
Line 1: ﻿"Data Source","World Development Indicators",
Line 2: 
Line 3: "Last Updated Date","2026-07-13",
Line 4: 
Line 5: "Country Name","Country Code","Indicator Name","Indicator Code","1960","1961","1962","1963","1964","1965","1966","1967","1968","1969","1970","1971","1972","1973","1974","1975","1976","1977","1978","1979","1980","1981","1982","1983","1984","1985","1986","1987","1988","1989","1990","1991","1992","1993","1994","1995","1996","1997","1998","1999","2000","2001","2002","2003","2004","2005","2006","2007","2008","2009","2010","2011","2012","2013","2014","2015","2016","2017","2018","2019","2020","2021","2022","2023","2024","2025",

Raw World Bank Shape (with skiprows=4): (265, 71)


#### **Filter Cohort & Inspect Schemas**

In [3]:
# 1. Filter Ember data for the 10-country cohort
print("Ember columns:", ember_raw.columns.tolist()[:10])

# Assuming the country column in Ember is 'Area' or 'Country'
country_col_ember = 'Area' if 'Area' in ember_raw.columns else 'Country'

ember_filtered = ember_raw[ember_raw[country_col_ember].isin(TARGET_COUNTRIES)].copy()
print(f"\nFiltered Ember Shape: {ember_filtered.shape}")
print("Countries found in Ember:", ember_filtered[country_col_ember].unique())

# 2. Filter World Bank data for the 10-country cohort
print("\nWorld Bank columns snippet:", wb_raw.columns.tolist()[:6])

wb_filtered = wb_raw[wb_raw['Country Name'].isin(TARGET_COUNTRIES)].copy()
print(f"Filtered World Bank Shape: {wb_filtered.shape}")
print("Countries found in World Bank:", wb_filtered['Country Name'].unique())
print("Available World Bank Indicators:", wb_filtered['Indicator Name'].unique())

Ember columns: ['Area', 'ISO 3 code', 'Year', 'Area type', 'Electricity source', 'Is aggregated source', 'Generation (TWh)', 'Generation YoY change (TWh)', 'Generation YoY change (%)', 'Share of generation (%)']

Filtered Ember Shape: (4064, 24)
Countries found in Ember: ['Egypt' 'Ethiopia' 'Ghana' 'Kenya' 'Morocco' 'Nigeria' 'Rwanda'
 'South Africa' 'Uganda']

World Bank columns snippet: ['Country Name', 'Country Code', 'Indicator Name', 'Indicator Code', '1960', '1961']
Filtered World Bank Shape: (9, 71)
Countries found in World Bank: ['Ethiopia' 'Ghana' 'Kenya' 'Morocco' 'Nigeria' 'Rwanda' 'Tanzania'
 'Uganda' 'South Africa']
Available World Bank Indicators: ['Access to electricity (% of population)']


#### **Fix Country Name Variations & Inspect Indicators**

In [4]:
# 1. Check all unique country names in raw Ember and World Bank to find naming variations
print("--- Checking All Unique Country Names in Raw Files ---")
print("Ember sample countries containing 'Tanzanian' or 'Tanzania' or 'Egypt':")
print(ember_raw[ember_raw['Area'].str.contains("Tanzania|Egypt", case=False, na=False)]['Area'].unique())

print("\nWorld Bank sample countries containing 'Tanzania' or 'Egypt':")
print(wb_raw[wb_raw['Country Name'].str.contains("Tanzania|Egypt", case=False, na=False)]['Country Name'].unique())

# 2. Map and standardize country names so all 10 match perfectly across datasets
name_mapping = {
    "Tanzania, United Republic of": "Tanzania",
    "Egypt, Arab Rep.": "Egypt"
}

ember_raw['Clean_Country'] = ember_raw['Area'].replace(name_mapping)
wb_raw['Clean_Country'] = wb_raw['Country Name'].replace(name_mapping)

# Filter with standardized names
ember_filtered = ember_raw[ember_raw['Clean_Country'].isin(TARGET_COUNTRIES)].copy()
wb_filtered = wb_raw[wb_raw['Clean_Country'].isin(TARGET_COUNTRIES)].copy()

print(f"\nSuccessfully matched Ember countries ({ember_filtered['Clean_Country'].nunique()}/10):")
print(sorted(ember_filtered['Clean_Country'].unique()))

print(f"\nSuccessfully matched World Bank countries ({wb_filtered['Clean_Country'].nunique()}/10):")
print(sorted(wb_filtered['Clean_Country'].unique()))

--- Checking All Unique Country Names in Raw Files ---
Ember sample countries containing 'Tanzanian' or 'Tanzania' or 'Egypt':
['Egypt' 'Tanzania (the United Republic of)']

World Bank sample countries containing 'Tanzania' or 'Egypt':
['Egypt, Arab Rep.' 'Tanzania']

Successfully matched Ember countries (9/10):
['Egypt', 'Ethiopia', 'Ghana', 'Kenya', 'Morocco', 'Nigeria', 'Rwanda', 'South Africa', 'Uganda']

Successfully matched World Bank countries (10/10):
['Egypt', 'Ethiopia', 'Ghana', 'Kenya', 'Morocco', 'Nigeria', 'Rwanda', 'South Africa', 'Tanzania', 'Uganda']


#### **Fix Tanzania Mapping & Melt World Bank Data**

In [5]:
# 1. Update name mapping with the exact Ember string for Tanzania
name_mapping = {
    "Tanzania (the United Republic of)": "Tanzania",
    "Egypt, Arab Rep.": "Egypt"
}

ember_raw['Clean_Country'] = ember_raw['Area'].replace(name_mapping)
wb_raw['Clean_Country'] = wb_raw['Country Name'].replace(name_mapping)

# Filter both datasets
ember_filtered = ember_raw[ember_raw['Clean_Country'].isin(TARGET_COUNTRIES)].copy()
wb_filtered = wb_raw[wb_raw['Clean_Country'].isin(TARGET_COUNTRIES)].copy()

print(f"Matched Ember countries ({ember_filtered['Clean_Country'].nunique()}/10):")
print(sorted(ember_filtered['Clean_Country'].unique()))

print(f"\nMatched World Bank countries ({wb_filtered['Clean_Country'].nunique()}/10):")
print(sorted(wb_filtered['Clean_Country'].unique()))

# 2. Melt World Bank data from wide to long format
year_columns = [col for col in wb_filtered.columns if col.isdigit()]

wb_long = wb_filtered.melt(
    id_vars=['Clean_Country', 'Country Code', 'Indicator Name', 'Indicator Code'],
    value_vars=year_columns,
    var_name='Year',
    value_name='Value'
)

# Convert Year to integer and filter for recent years
wb_long['Year'] = wb_long['Year'].astype(int)
wb_long = wb_long[(wb_long['Year'] >= 2000) & (wb_long['Value'].notna())]

print("\nMelted World Bank Data Shape:", wb_long.shape)
print(wb_long.head(3))

Matched Ember countries (10/10):
['Egypt', 'Ethiopia', 'Ghana', 'Kenya', 'Morocco', 'Nigeria', 'Rwanda', 'South Africa', 'Tanzania', 'Uganda']

Matched World Bank countries (10/10):
['Egypt', 'Ethiopia', 'Ghana', 'Kenya', 'Morocco', 'Nigeria', 'Rwanda', 'South Africa', 'Tanzania', 'Uganda']

Melted World Bank Data Shape: (250, 6)
    Clean_Country Country Code                           Indicator Name  \
400         Egypt          EGY  Access to electricity (% of population)   
401      Ethiopia          ETH  Access to electricity (% of population)   
402         Ghana          GHA  Access to electricity (% of population)   

     Indicator Code  Year  Value  
400  EG.ELC.ACCS.ZS  2000   97.7  
401  EG.ELC.ACCS.ZS  2000   12.7  
402  EG.ELC.ACCS.ZS  2000   43.7  


#### **Inspect Ember Dataset Structure & Technologies**

In [6]:
# 1. Inspect the time range and technology sources in filtered Ember data
print(f"Ember Data Shape: {ember_filtered.shape}")
print(f"Years available in Ember: {ember_filtered['Year'].min()} to {ember_filtered['Year'].max()}")

# 2. Check unique electricity sources / technologies tracked
print("\nAvailable Electricity Sources in Ember:")
print(ember_filtered['Electricity source'].unique())

# 3. Check for any missing values in key columns
print("\nMissing values check in Ember filtered:")
print(ember_filtered[['Clean_Country', 'Year', 'Electricity source', 'Generation (TWh)', 'Share of generation (%)']].isnull().sum())

# 4. Preview sample rows
print("\nEmber Sample Rows:")
print(ember_filtered[['Clean_Country', 'Year', 'Electricity source', 'Generation (TWh)']].head(5))

Ember Data Shape: (4489, 25)
Years available in Ember: 1985 to 2025

Available Electricity Sources in Ember:
['Wind' 'Hydro' 'Nuclear' 'Coal' 'Gas' 'Other fossil' 'Wind and solar'
 'Hydro, bioenergy and other renewables' 'Renewables' 'Clean' 'Fossil'
 'Total generation' 'Demand' 'Solar' 'Bioenergy' 'Net imports'
 'Other renewables']

Missing values check in Ember filtered:
Clean_Country              0
Year                       0
Electricity source         0
Generation (TWh)           0
Share of generation (%)    0
dtype: int64

Ember Sample Rows:
      Clean_Country  Year Electricity source  Generation (TWh)
27350         Egypt  1985               Wind              0.00
27351         Egypt  1985              Hydro              9.15
27352         Egypt  1985            Nuclear              0.00
27353         Egypt  1985               Coal              0.00
27354         Egypt  1985                Gas              9.24


#### **Export Cleaned Datasets to Processed Folder**

In [9]:
# 1. Export cleaned Ember data
ember_filtered.to_csv("../data/processed/ember_cleaned.csv", index=False)
print("Saved: data/processed/ember_cleaned.csv")

# 2. Export cleaned World Bank melted data
wb_long.to_csv("../data/processed/world_bank_cleaned.csv", index=False)
print("Saved: data/processed/world_bank_cleaned.csv")

Saved: data/processed/ember_cleaned.csv
Saved: data/processed/world_bank_cleaned.csv
